In [1]:
# cell 1 immports
import sys
sys.path.append('..')
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_loading import *
from src.preprocessing import *
from src.train import *
from src.visualization import *
from src.evaluate import evaluate_model
import warnings
from sklearn.exceptions import UndefinedMetricWarning
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)

In [9]:
# Cell 2: Load config and data
config = load_config('../configs/config.yaml')
data_dir = config['data']['data_dir']
s2_dir = os.path.join(data_dir, config['data']['s2_dir'])
ann_dir = os.path.join(data_dir, config['data']['ann_dir'])

In [3]:
# Cell 3: Create splits
train_ids, val_ids, test_ids = get_data_splits(s2_dir)
save_splits(train_ids, val_ids, test_ids)
print(f"Train: {len(train_ids)}, Val: {len(val_ids)}, Test: {len(test_ids)}")

Train: 71, Val: 15, Test: 16


In [4]:
# Cell 4: Build dataset
X_train, y_train = build_dataset(
    train_ids, s2_dir, ann_dir,
    max_patches=config['sampling']['max_patches'],
    pixels_per_patch=config['sampling']['pixels_per_patch']
)
print(f"Training data: {X_train.shape}")

Training data: (30000, 40)


In [5]:
# Check that classes 0 and 19 are excluded
unique_classes = np.unique(y_train)
print(f"Classes in training data: {sorted(unique_classes)}")
print(f"Class 0 excluded: {0 not in unique_classes}")
print(f"Class 19 excluded: {19 not in unique_classes}")

Classes in training data: [np.uint8(1), np.uint8(2), np.uint8(3), np.uint8(4), np.uint8(5), np.uint8(6), np.uint8(7), np.uint8(8), np.uint8(10), np.uint8(12), np.uint8(14), np.uint8(15), np.uint8(17), np.uint8(18)]
Class 0 excluded: True
Class 19 excluded: True


In [6]:
# Cell 5: Build validation dataset (all pixels)
X_val, y_val = build_dataset(val_ids, s2_dir, ann_dir)
print(f"Validation data: {X_val.shape}")

Validation data: (160243, 40)


In [10]:
# Cell 6: Train LightGBM
model = train_model(X_train, y_train, config)
save_model(model)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.029993 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 10200
[LightGBM] [Info] Number of data points in the train set: 30000, number of used features: 40
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGB

In [11]:
# Cell 7: Evaluate
y_pred, accuracy, report, cm = evaluate_model(model, X_val, y_val)

In [35]:
print(f"Accuracy: {accuracy:.4f}")

print("\nClassification Report:")
print(classification_report(y_val, y_pred, zero_division=0))
print(f"Accuracy: {accuracy:.4f}")

print("\nClassification Report:")
print(classification_report(y_val, y_pred, zero_division=0))

# Save to file (if evaluate_model didn't save)
os.makedirs('../outputs/metrics', exist_ok=True)
with open('../outputs/metrics/classification_report.txt', 'w') as f:
    f.write(f"Accuracy: {accuracy:.4f}\n\n")
    f.write(classification_report(y_val, y_pred, zero_division=0))

Accuracy: 0.6072

Classification Report:
              precision    recall  f1-score   support

           1       0.73      0.82      0.77     35592
           2       0.49      0.71      0.58     24217
           3       0.51      0.60      0.55     35716
           4       0.47      0.13      0.21     11473
           5       0.90      0.79      0.84     24589
           6       0.00      0.00      0.00      1434
           7       0.00      0.00      0.00       687
           8       0.00      0.00      0.00         0
          10       0.00      0.00      0.00      1641
          12       0.00      0.00      0.00         0
          14       0.25      0.13      0.17      2859
          15       0.52      0.38      0.44     22035
          17       0.00      0.00      0.00         0
          18       0.00      0.00      0.00         0

    accuracy                           0.61    160243
   macro avg       0.28      0.25      0.25    160243
weighted avg       0.60      0.61      

In [12]:
# Cell 8: Plot confusion matrix
class_names = ['Meadow', 'Soft winter wheat', 'Corn', 'Winter barley', 
               'Winter rapeseed', 'Spring barley', 'Sunflower', 'Grapevine',
               'Beet', 'Winter triticale', 'Winter durum wheat', 
               'Fruits/vegetables/flowers', 'Potatoes', 'Leguminous fodder',
               'Soybeans', 'Orchard', 'Mixed cereal', 'Sorghum']
# Map class IDs to names (1-18)
cm_classes = sorted(np.unique(y_val))
cm_labels = [class_names[c-1] for c in cm_classes]
plot_confusion_matrix(cm, cm_labels, '../outputs/figures/confusion_matrix.png')


In [13]:
# Cell 9: Plot class distribution
class_counts = {c: np.sum(y_train == c) for c in np.unique(y_train)}
plot_class_distribution(class_counts, '../outputs/figures/class_distribution.png')


In [14]:
# Cell 10: Feature importance
plot_feature_importance(model, '../outputs/figures/feature_importance.png')

In [15]:
# Cell 11: Visualize test patch
test_patch = test_ids[0]
s2_test, target_test = load_patch(test_patch, s2_dir, ann_dir)

# RGB
rgb = s2_test[0, [2,1,0], :, :].transpose(1,2,0)
rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)

# Predict
X_test, _, mask = prepare_data(s2_test, target_test)
preds = model.predict(X_test)

# Create prediction maps
pred_map = np.zeros_like(target_test[0])
pred_map[mask] = preds
pred_map_smooth = apply_spatial_smoothing(pred_map, window_size=5)

# Plot comparison
plot_prediction_comparison(
    rgb, target_test[0], pred_map, pred_map_smooth,
    '../outputs/figures/test_patch_comparison.png'
)

Saved visualization: ../outputs/figures/test_patch_comparison.png


In [16]:
# Cell 12: Save predictions
np.save('../outputs/predictions/test_patch_predictions.npy', pred_map)
np.save('../outputs/predictions/test_patch_smoothed.npy', pred_map_smooth)

print("Done! Check outputs/ directory for results.")

Done! Check outputs/ directory for results.
